In [1]:
import os
from glob import glob
from PIL import Image
import numpy as np
from ultralytics import YOLO
from tqdm import tqdm
import cv2 # 마스크 후처리 및 이미지 합성용

def process_dog_segmentation_with_yolo(input_dir, output_dir):
    """
    폴더 내 모든 이미지에서 강아지만 분리하여 배경을 제거하고 저장합니다.
    YOLOv8 Segmentation 모델과 MPS(Apple Silicon GPU) 가속을 사용합니다.
    """
    # 1. 모델 로드
    # yolov8n-seg.pt: 가장 작고 빠른 모델
    # 더 높은 정확도를 원하면 'yolov8s-seg.pt'나 'yolov8m-seg.pt' 등을 사용하세요.
    print("YOLOv8 Segmentation 모델을 로드합니다...")
    try:
        model = YOLO('yolov8n-seg.pt') # 첫 실행 시 모델 가중치가 자동으로 다운로드됩니다.
        print("✅ 모델 로드 완료.")
    except Exception as e:
        print(f"❌ 모델 로드 중 오류 발생: {e}")
        print("인터넷 연결을 확인하거나, 'pip install ultralytics'가 제대로 설치되었는지 확인해주세요.")
        return

    # 2. 처리할 이미지 파일 경로 탐색
    image_paths = []
    supported_extensions = ["*.jpg", "*.jpeg", "*.png", "*.webp"]
    for ext in supported_extensions:
        image_paths.extend(glob(os.path.join(input_dir, '**', ext), recursive=True))

    if not image_paths:
        print(f"⚠️ '{input_dir}' 폴더에 처리할 이미지가 없습니다. 경로를 확인해주세요.")
        return

    # 3. 각 이미지에 대해 분리 작업 수행
    for img_path in tqdm(image_paths, desc="이미지 처리 중"):
        try:
            # YOLO 모델 추론 (MPS 디바이스 사용)
            # verbose=False 옵션으로 불필요한 로그 출력을 줄입니다.
            results = model(img_path, device='mps', conf=0.1, verbose=False)

            # 원본 이미지 로드
            original_img_pil = Image.open(img_path).convert("RGBA")
            original_img_np = np.array(original_img_pil)

            # 모든 강아지 마스크를 합칠 빈 마스크 생성
            combined_dog_mask = np.zeros(original_img_np.shape[:2], dtype=np.uint8)
            found_dog = False

            for r in results:
                # 결과에 마스크와 박스 정보가 있는지 확인
                if r.masks is not None and r.boxes is not None:
                    for i, box in enumerate(r.boxes):
                        class_id = int(box.cls)
                        class_name = model.names[class_id]

                        # 클래스가 'dog'일 경우에만 마스크 처리
                        if class_name == 'dog':
                            found_dog = True
                            mask_tensor = r.masks.data[i]

                            # 마스크 크기를 원본 이미지 크기에 맞게 조정
                            mask_resized = cv2.resize(
                                mask_tensor.cpu().numpy(),
                                (original_img_np.shape[1], original_img_np.shape[0]),
                                interpolation=cv2.INTER_LINEAR
                            )

                            # 마스크 이진화 (0 또는 255) 및 합치기
                            mask_binary = (mask_resized > 0.5).astype(np.uint8) * 255
                            combined_dog_mask = cv2.bitwise_or(combined_dog_mask, mask_binary)

            # 강아지가 감지된 경우에만 배경 제거 수행
            if found_dog:
                # 최종 합쳐진 마스크를 사용해 원본 이미지의 알파 채널(투명도) 조작
                final_img_np = original_img_np.copy()
                # 마스크가 0인 부분(배경)을 투명하게 만듦
                final_img_np[:, :, 3] = np.where(combined_dog_mask == 255, final_img_np[:, :, 3], 0)
                result_img_pil = Image.fromarray(final_img_np, 'RGBA')
            else:
                # 강아지가 없으면 원본을 그대로 사용 (배경 제거 안 함)
                result_img_pil = original_img_pil.convert("RGB") # 알파채널 제거 후 저장
                tqdm.write(f"ℹ️ '{os.path.basename(img_path)}'에서 강아지를 찾지 못했습니다.")


            # 결과 이미지 저장 (폴더 구조 유지, PNG로 저장하여 투명도 보존)
            relative_path = os.path.relpath(img_path, input_dir)
            output_filename = os.path.splitext(relative_path)[0] + ".png"
            output_path = os.path.join(output_dir, output_filename)

            os.makedirs(os.path.dirname(output_path), exist_ok=True)
            result_img_pil.save(output_path)

        except Exception as e:
            tqdm.write(f"❌ 오류 발생: '{os.path.basename(img_path)}' 처리 중 - {e}")

    print(f"\n✨ 모든 작업이 완료되었습니다! 결과는 '{output_dir}' 폴더에 저장되었습니다.")

if __name__ == '__main__':
    # --- ⚙️ 설정 ---
    # 1. 원본 이미지가 있는 폴더 경로
    input_folder = "./dog_images_output"

    # 2. 결과물을 저장할 폴더 경로
    output_folder = "./dog_images_output_final"
    # ----------------

    process_dog_segmentation_with_yolo(input_folder, output_folder)

YOLOv8 Segmentation 모델을 로드합니다...
✅ 모델 로드 완료.


이미지 처리 중:   0%|          | 0/16843 [00:00<?, ?it/s]/var/folders/x_/xf7w4xhx6kl03__yxdgdfl9m0000gn/T/ipykernel_50845/3809711709.py:80: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  result_img_pil = Image.fromarray(final_img_np, 'RGBA')
이미지 처리 중:   0%|          | 2/16843 [00:01<2:27:38,  1.90it/s]      

ℹ️ 'Toy Poodle294.png'에서 강아지를 찾지 못했습니다.


이미지 처리 중:   0%|          | 3/16843 [00:01<1:56:57,  2.40it/s]      

ℹ️ 'Toy Poodle56.png'에서 강아지를 찾지 못했습니다.


...

ℹ️ 'golden retriever330.png'에서 강아지를 찾지 못했습니다.


이미지 처리 중: 100%|█████████▉| 16838/16843 [17:23<00:00, 15.08it/s]      

ℹ️ 'golden retriever293.png'에서 강아지를 찾지 못했습니다.


이미지 처리 중: 100%|██████████| 16843/16843 [17:23<00:00, 16.14it/s]      

ℹ️ 'golden retriever708.png'에서 강아지를 찾지 못했습니다.
ℹ️ 'golden retriever913.png'에서 강아지를 찾지 못했습니다.
ℹ️ 'golden retriever907.png'에서 강아지를 찾지 못했습니다.

✨ 모든 작업이 완료되었습니다! 결과는 './dog_images_output_final' 폴더에 저장되었습니다.
